# 🔢 Notebook 2: NumPy & The Math of Neural Networks
### Vectors, Matrices, and Why They Run the World of AI

---

**Prerequisites:** Notebook 1 (Python Basics)

**What you'll learn:**
1. What NumPy is and why it matters
2. Vectors — the language of AI
3. Matrices — the engine of neural networks
4. Dot products and matrix multiplication
5. The concept of embeddings (how language models see words)
6. Cosine similarity — how we measure meaning
7. Mini-project: Build a real vectorized neural network layer

---

**Why this matters for interpretability:**
Interpretability research is fundamentally about analyzing vectors and matrices. When researchers say "this neuron represents the concept of Paris" or "these two features are similar," they are doing linear algebra on the model's activations. This notebook is the mathematical foundation of everything.

In [ ]:
# First, import NumPy. This is the first line of almost every ML script.
import numpy as np
import matplotlib.pyplot as plt  # for plotting

print("NumPy version:", np.__version__)
print("Libraries loaded! Let's go.")

---
## Section 1: Why NumPy?

In Notebook 1, we used Python lists and loops. That works, but it's slow.

NumPy replaces slow Python loops with fast C-level operations. The difference is dramatic:

In [ ]:
import time

size = 1_000_000  # 1 million numbers

# Python list way
python_list = list(range(size))
start = time.time()
result = [x * 2 for x in python_list]
python_time = time.time() - start

# NumPy way
numpy_array = np.arange(size)
start = time.time()
result = numpy_array * 2
numpy_time = time.time() - start

print(f"Python list: {python_time*1000:.2f} ms")
print(f"NumPy array: {numpy_time*1000:.2f} ms")
print(f"NumPy is {python_time/numpy_time:.0f}x faster!")
print("\nThis matters when your model has billions of operations per forward pass.")

---
## Section 2: Vectors — The Language of AI

A **vector** is just an ordered list of numbers. But in AI, vectors are how the model *represents everything*:
- A word is a vector (its "embedding")
- An image is a vector (its pixel values)
- A neuron's activation is a vector
- A "feature" found by interpretability research is a *direction* in vector space

The key insight: **similar things have similar vectors.**  
In a trained language model, the vector for "king" is close to the vector for "queen".

In [ ]:
# Creating vectors (1D arrays)
# In GPT-2, every word is represented as a 768-dimensional vector.
# We'll use small examples here for clarity.

# Imagine a 3D space where axes are: [royalty, gender, power]
king   = np.array([0.9, 0.1, 0.9])   # high royalty, male, high power
queen  = np.array([0.9, 0.9, 0.9])   # high royalty, female, high power
man    = np.array([0.1, 0.1, 0.5])   # low royalty, male, medium power
woman  = np.array([0.1, 0.9, 0.5])   # low royalty, female, medium power

print("king  :", king)
print("queen :", queen)
print("man   :", man)
print("woman :", woman)

In [ ]:
# FAMOUS DEMO: king - man + woman ≈ queen
# This actually works in real word embeddings!

result = king - man + woman
print("king - man + woman =", result)
print("queen              =", queen)
print("\nThey're very close! This is the famous word analogy result.")
print("Real models learn this structure automatically from text.")

In [ ]:
# Basic vector operations
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print("a + b =", a + b)        # element-wise addition
print("a * 2 =", a * 2)        # scalar multiplication
print("a * b =", a * b)        # element-wise multiplication

# Vector magnitude (length) — how 'big' is the vector?
magnitude = np.linalg.norm(a)
print(f"\n|a| = {magnitude:.4f}")

# Unit vector (direction only, magnitude = 1)
unit_a = a / magnitude
print(f"unit a = {unit_a}")
print(f"|unit a| = {np.linalg.norm(unit_a):.4f}  (should be 1.0)")

---
## Section 3: The Dot Product — Most Important Operation in AI

The **dot product** multiplies corresponding elements and sums them:

$$\mathbf{a} \cdot \mathbf{b} = a_1 b_1 + a_2 b_2 + a_3 b_3 = \sum_i a_i b_i$$

Why it matters:
- **Attention mechanisms** in transformers are built on dot products
- The dot product measures **how aligned** two vectors are
- Every neuron computation is a dot product of inputs with weights

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# Three equivalent ways to compute dot product:
dot1 = a[0]*b[0] + a[1]*b[1] + a[2]*b[2]   # manual
dot2 = np.dot(a, b)                           # NumPy function
dot3 = a @ b                                  # @ operator (modern Python)

print(f"Manual:  {dot1}")
print(f"np.dot:  {dot2}")
print(f"@ operator: {dot3}")
print("\nAll the same! Use @ in practice — it's the clearest.")

In [ ]:
# The dot product measures ALIGNMENT
# Positive = vectors point in same direction
# Zero = vectors are perpendicular (unrelated)
# Negative = vectors point in opposite directions

v1 = np.array([1.0, 0.0, 0.0])
same_dir   = np.array([2.0, 0.0, 0.0])
perpendic  = np.array([0.0, 1.0, 0.0])
opposite   = np.array([-1.0, 0.0, 0.0])

print(f"Same direction dot product:    {v1 @ same_dir:.1f}  (positive — aligned)")
print(f"Perpendicular dot product:     {v1 @ perpendic:.1f}  (zero — unrelated)")
print(f"Opposite direction dot product: {v1 @ opposite:.1f}  (negative — opposite)")

print("\n💡 In attention, the model uses dot products to decide")
print("   which words to 'attend to' when reading a sentence.")

---
## Section 4: Matrices — The Engine of Neural Networks

A **matrix** is a 2D array of numbers (rows × columns).

In neural networks:
- The **weight matrix** of a layer defines how it transforms its inputs
- Running a batch of inputs through a layer = one matrix multiplication
- The entire forward pass of a transformer is a sequence of matrix multiplications

For interpretability: the weight matrices ARE the model. Understanding them is understanding the model.

In [ ]:
# Creating matrices
# A weight matrix: 3 inputs → 2 outputs
# Each ROW is the weights for one output neuron
W = np.array([
    [0.5, -0.3,  0.8],   # weights for output neuron 1
    [0.2,  0.9, -0.4],   # weights for output neuron 2
])

print("Weight matrix W:")
print(W)
print(f"\nShape: {W.shape}  → ({W.shape[0]} output neurons, {W.shape[1]} input connections)")

In [ ]:
# Matrix × Vector = the fundamental neural network operation
# W @ x transforms input vector x through the weight matrix W

x = np.array([1.0, 2.0, 3.0])  # input vector (3 features)

output = W @ x  # matrix-vector multiply
print(f"Input x: {x}  (shape: {x.shape})")
print(f"Weight W shape: {W.shape}")
print(f"Output W@x: {output}  (shape: {output.shape})")

# Verify manually:
manual_out1 = 0.5*x[0] + (-0.3)*x[1] + 0.8*x[2]
manual_out2 = 0.2*x[0] + 0.9*x[1] + (-0.4)*x[2]
print(f"\nManual check: [{manual_out1:.1f}, {manual_out2:.1f}]  ✓")

In [ ]:
# Matrix × Matrix — process a whole BATCH at once
# In ML, we always process many examples simultaneously (a "batch")
# Each row of X is one example

X_batch = np.array([
    [1.0, 2.0, 3.0],   # example 1
    [0.5, 1.5, 2.5],   # example 2
    [2.0, 0.0, 1.0],   # example 3
])  # shape: (3 examples, 3 features)

# To apply W to all examples: X @ W.T
# (we transpose W because we want (batch, in) @ (in, out) = (batch, out))
batch_output = X_batch @ W.T

print(f"Batch input shape:  {X_batch.shape}  (3 examples × 3 features)")
print(f"Weight matrix shape: {W.shape}  (2 outputs × 3 inputs)")
print(f"Batch output shape: {batch_output.shape}  (3 examples × 2 outputs)")
print("\nBatch output:")
print(batch_output)

---
## Section 5: Cosine Similarity — How We Measure Meaning

**Cosine similarity** is the standard way to measure how similar two vectors are in AI.

It measures the *angle* between two vectors, not their magnitude:
- **1.0** = identical direction (same concept)
- **0.0** = perpendicular (unrelated)
- **-1.0** = opposite directions (opposite concepts)

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}| \cdot |\mathbf{b}|}$$

**In interpretability:** when researchers find a "feature" (a direction in activation space), they use cosine similarity to check whether two features are related, or whether a feature found in one model also exists in another model.

In [ ]:
def cosine_similarity(a, b):
    """Measure how similar two vectors are (range: -1 to 1)."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot / (norm_a * norm_b)


# Simulated word embeddings (in reality these are 768-dim vectors)
embeddings = {
    "Paris":   np.array([0.9, 0.1, 0.8, 0.2]),
    "London":  np.array([0.8, 0.1, 0.7, 0.3]),
    "France":  np.array([0.8, 0.2, 0.9, 0.1]),
    "dog":     np.array([0.1, 0.8, 0.0, 0.9]),
    "cat":     np.array([0.2, 0.9, 0.0, 0.8]),
    "banana":  np.array([0.1, 0.5, 0.1, 0.2]),
}

# Compare Paris to everything else
print("Cosine similarity with 'Paris':")
print("-" * 35)
paris = embeddings["Paris"]
for word, vec in embeddings.items():
    if word != "Paris":
        sim = cosine_similarity(paris, vec)
        bar = "█" * int(sim * 20)
        print(f"  Paris vs {word:8s}: {sim:.3f}  {bar}")

In [ ]:
# Visualize all pairwise similarities as a heatmap
# This is the kind of analysis interpretability researchers do!

words = list(embeddings.keys())
n = len(words)
sim_matrix = np.zeros((n, n))

for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i, j] = cosine_similarity(embeddings[w1], embeddings[w2])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_yticklabels(words)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Word Embedding Similarity Matrix\n(clusters reveal related concepts)')

# Add values in cells
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print("\nNotice: Paris/London/France cluster together, dog/cat cluster together!")

---
## Section 6: Superposition — A Taste of Interpretability

This is a simplified but real demonstration of the core problem in interpretability.

**Superposition:** a network with fewer neurons than features it wants to represent will *overlap* features, encoding multiple concepts in each neuron.

Here we simulate it directly.

In [ ]:
np.random.seed(42)

# Imagine 5 'true' features the model wants to represent
# (e.g., "is_capital_city", "is_animal", "is_food", "is_positive", "is_abstract")
n_features = 5
n_neurons  = 2   # but we only have 2 neurons!

# Each feature gets assigned a random direction in 2D neuron space
# (In reality this is learned by the network, not randomly assigned)
feature_directions = np.random.randn(n_features, n_neurons)
# Normalize each direction to unit length
feature_directions /= np.linalg.norm(feature_directions, axis=1, keepdims=True)

feature_names = ["capital_city", "animal", "food", "positive", "abstract"]

print("Feature directions in 2D neuron space:")
print(f"{'Feature':<15} {'Neuron 1':>10} {'Neuron 2':>10}")
print("-" * 37)
for name, direction in zip(feature_names, feature_directions):
    print(f"{name:<15} {direction[0]:>10.3f} {direction[1]:>10.3f}")

print("\n5 features squeezed into 2 neurons — this IS superposition!")

In [ ]:
# Visualize the feature directions
fig, ax = plt.subplots(figsize=(7, 7))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for i, (name, direction) in enumerate(zip(feature_names, feature_directions)):
    ax.arrow(0, 0, direction[0], direction[1],
             head_width=0.05, head_length=0.05,
             fc=colors[i], ec=colors[i], linewidth=2)
    ax.text(direction[0]*1.15, direction[1]*1.15, name,
            ha='center', va='center', fontsize=10, color=colors[i], fontweight='bold')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Neuron 1 activation', fontsize=12)
ax.set_ylabel('Neuron 2 activation', fontsize=12)
ax.set_title('Superposition: 5 features in 2D neuron space\n'
             'Each arrow = a feature direction', fontsize=12)
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', linewidth=1)
ax.add_patch(circle)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("\nThis is why a single neuron can respond to multiple, unrelated concepts.")
print("This is the CORE PROBLEM that interpretability research is trying to solve!")

In [ ]:
# The interference problem:
# When we look at a single neuron, it's a mix of ALL the features

# Say the input activates the 'capital_city' and 'positive' features
# (like the word "Paris" might)
active_features = {"capital_city": 0.9, "positive": 0.6}

# What do the 2 neurons actually show?
neuron_activations = np.zeros(n_neurons)
for feature_name, strength in active_features.items():
    idx = feature_names.index(feature_name)
    neuron_activations += strength * feature_directions[idx]

print("Input: 'Paris' activates capital_city=0.9, positive=0.6")
print(f"Neuron 1 shows: {neuron_activations[0]:.3f}")
print(f"Neuron 2 shows: {neuron_activations[1]:.3f}")
print("\nProblem: If you just look at Neuron 1 or Neuron 2,")
print("you can't tell WHICH feature caused the activation!")
print("\nSolution: Sparse Autoencoders (SAEs) — covered in Notebook 4!")

---
## Section 7: Mini-Project — A Full Neural Network Layer

Now let's build a proper vectorized neural network layer using everything you've learned.
This is essentially what PyTorch's `nn.Linear` does under the hood.

In [ ]:
class LinearLayer:
    """
    A single fully-connected (linear) layer.
    y = x @ W.T + b
    
    This is the core building block of neural networks.
    In transformers, there are many of these (the MLP layers).
    """
    
    def __init__(self, input_size, output_size):
        # Initialize weights with small random values (He initialization)
        scale = np.sqrt(2.0 / input_size)
        self.W = np.random.randn(output_size, input_size) * scale
        self.b = np.zeros(output_size)
        self.input_size = input_size
        self.output_size = output_size
    
    def forward(self, x):
        """Transform input x through this layer."""
        return x @ self.W.T + self.b
    
    def num_params(self):
        return self.W.size + self.b.size
    
    def __repr__(self):
        return f"LinearLayer({self.input_size} → {self.output_size}, params={self.num_params():,})"


def relu(x):
    return np.maximum(0, x)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))  # subtract max for stability
    return e / e.sum(axis=-1, keepdims=True)


# Build a simple 3-layer MLP
layer1 = LinearLayer(784, 256)   # e.g., flattened 28x28 image → 256 hidden
layer2 = LinearLayer(256, 128)   # 256 → 128
layer3 = LinearLayer(128, 10)    # 128 → 10 classes (e.g., digit 0-9)

print("Network architecture:")
print(f"  {layer1}")
print(f"  {layer2}")
print(f"  {layer3}")
total = layer1.num_params() + layer2.num_params() + layer3.num_params()
print(f"\nTotal parameters: {total:,}")

In [ ]:
# Forward pass — run a batch of 32 images through the network
np.random.seed(0)
batch_size = 32
x = np.random.randn(batch_size, 784)  # 32 fake images

# Pass through each layer with ReLU activation
h1 = relu(layer1.forward(x))
h2 = relu(layer2.forward(h1))
logits = layer3.forward(h2)    # raw scores
probs = softmax(logits)         # probabilities

print("Forward pass:")
print(f"  Input:   {x.shape}")
print(f"  After L1: {h1.shape}  (ReLU applied)")
print(f"  After L2: {h2.shape}  (ReLU applied)")
print(f"  Logits:  {logits.shape}")
print(f"  Probs:   {probs.shape}")

print(f"\nFirst example's class probabilities:")
for digit, prob in enumerate(probs[0]):
    bar = "█" * int(prob * 50)
    print(f"  Digit {digit}: {prob:.3f}  {bar}")
print(f"  Sum: {probs[0].sum():.6f}  (should be 1.0)")

In [ ]:
# INTERPRETABILITY HOOK: What are the activations?
# In real interpretability work, you capture h1, h2 etc. and analyze them.
# Which neurons fired? What do they represent?

print("Analyzing hidden layer activations (interpretability style!):")
print("=" * 55)

# ReLU means some neurons are 'dead' (output 0) for a given input
h1_active = (h1 > 0).mean(axis=1)  # fraction of neurons active per example
h2_active = (h2 > 0).mean(axis=1)

print(f"Layer 1: avg {h1_active.mean():.1%} of neurons active per example")
print(f"Layer 2: avg {h2_active.mean():.1%} of neurons active per example")

# Find the most consistently active neurons in layer 1
neuron_activity = (h1 > 0).mean(axis=0)  # fraction of examples each neuron activates for
top_neurons = np.argsort(neuron_activity)[-5:][::-1]
print(f"\nMost consistently active neurons in layer 1:")
for n in top_neurons:
    print(f"  Neuron {n:3d}: active {neuron_activity[n]:.1%} of the time")

print("\n💡 Real interpretability: we'd now look at WHAT INPUT PATTERNS")
print("   cause these neurons to fire — that's how we find 'features'!")

---
### 🎯 Final Exercise: Explore Your Own Network

1. Create a different architecture: try `[784 → 512 → 256 → 10]`
2. Run a batch through it and capture all intermediate activations
3. For layer 1: plot a histogram of all activation values (hint: `plt.hist(h1.flatten(), bins=50)`)
4. What happens to activations after ReLU? (Compare the histogram of pre-ReLU vs post-ReLU values)

*This is exactly the kind of analysis interpretability researchers do when first exploring a model.*

In [ ]:
# YOUR CODE HERE


---
## 🎉 Notebook 2 Complete!

You now understand:
- ✅ Why NumPy is essential for ML
- ✅ Vectors — the fundamental representation in AI
- ✅ Dot products — the most important operation
- ✅ Matrix multiplication — the engine of neural networks
- ✅ Cosine similarity — how to measure conceptual similarity
- ✅ Superposition — the core challenge of interpretability
- ✅ Built a real neural network forward pass from scratch

**Next step:** Notebook 3 — Transformers from Scratch  
We'll build attention mechanisms and understand why interpretability focuses on them so heavily.